In [9]:
# set the directory
import os
path = '/result-3DHPE/base/mmPose-CNN'
os.chdir(path)
print(os.getcwd())
!ls

/result-3DHPE/base/mmPose-CNN
output_mmpose-cnn  reshaped_data_mmPoseCNN


In [6]:
def project_point_cloud_to_image(points, plane='xy', grid_size=16):
    """将雷达点云投影到图像"""
    if plane == 'xy':
        # 处理XY平面投影 (x∈[-1,1], y∈[0,3])
        x_coords = points[:, 0]
        y_coords = points[:, 1]
        # 归一化坐标
        x_norm = (x_coords - (-1.0)) / (1.0 - (-1.0))  # x坐标归一化到[0,1]
        y_norm = (y_coords - 0.0) / (3.0 - 0.0)        # y坐标归一化到[0,1]
        coords = np.stack([x_norm, y_norm], axis=1)
    else:
        # 处理XZ平面投影 (x∈[-1,1], z∈[-1,1])
        x_coords = points[:, 0]
        z_coords = points[:, 2]
        # 归一化坐标
        x_norm = (x_coords - (-1.0)) / (1.0 - (-1.0))  # x坐标归一化到[0,1]
        z_norm = (z_coords - (-1.0)) / (1.0 - (-1.0))  # z坐标归一化到[0,1]
        coords = np.stack([x_norm, z_norm], axis=1)
    
    intensity = points[:, 4]  # 反射强度
    
    # 将坐标映射到网格中
    grid_indices = (coords * (grid_size - 1)).astype(int)
    image = np.zeros((grid_size, grid_size, 3))
    
    for idx, (i, j) in enumerate(grid_indices):
        if 0 <= i < grid_size and 0 <= j < grid_size:
            image[i, j, 0] = coords[idx, 0]  # 归一化后的坐标值
            image[i, j, 1] = coords[idx, 1]
            image[i, j, 2] = intensity[idx]
    return image

def preprocess_all_data(radar_data, grid_size=16):
    """预先生成所有雷达数据的投影图像"""
    features = []
    for points in radar_data:
        xy_image = project_point_cloud_to_image(points, 'xy')
        xz_image = project_point_cloud_to_image(points, 'xz')
        features.append([xy_image, xz_image])
    return np.array(features)



In [8]:

import numpy as np 
# from keras.models import load_model
from tensorflow.keras.models import load_model
from keras.optimizers import Adam
import tensorflow as tf
from sklearn import metrics
import time
import pandas as pd

# 加载模型时传递自定义指标
model_path = 'output_mmpose-cnn/mmpose-cnn.h5' 
loaded_model = load_model(
    model_path,
    compile = False
)

# 验证模型加载成功
loaded_model.summary()

# 加载测试数据（假设已经加载，这里直接使用）
# 预处理所有数据
feature_test = np.load('reshaped_data_mmPoseCNN/feature_test.npy')
featuremap_test = preprocess_all_data(feature_test)

# 分离为两个输入分支
xy_test = featuremap_test[:, 0]
xz_test = featuremap_test[:, 1]
labels_test = np.load('reshaped_data_mmPoseCNN/labels_test.npy')
n_frames = labels_test.shape[0]
print('frames: ', n_frames)

# ---------------- 推理时间 ----------------
start_time = time.time()
# 使用模型进行预测
predictions = loaded_model.predict([xy_test, xz_test], batch_size=64, verbose=0)
end_time = time.time()
total_inference_time = end_time - start_time
avg_inference_time_per_frame = total_inference_time / n_frames
print('avg_inference_time_per_frame: ', avg_inference_time_per_frame)

# 保存预测结果到 .npy 文件
filtered_result_path = 'output_mmpose-cnn/mmpose-cnn_result.npy'
np.save(filtered_result_path, predictions)

# ---------------- MAE 计算 ----------------
# 分别计算 x, y, z 方向的 MAE（每个轴19个关节的所有帧误差）
x_mae = metrics.mean_absolute_error(labels_test[:, 0:19], predictions[:, 0:19], multioutput='raw_values')
y_mae = metrics.mean_absolute_error(labels_test[:, 19:38], predictions[:, 19:38], multioutput='raw_values')
z_mae = metrics.mean_absolute_error(labels_test[:, 38:57], predictions[:, 38:57], multioutput='raw_values')

# 将三个轴的误差拼接成一个 3 x 19 的矩阵，然后转置为 19 x 3
all_19_points_mae = np.concatenate((x_mae, y_mae, z_mae)).reshape(3, 19)
all_19_points_mae_transpose = all_19_points_mae.T
print("MAE for 19 joints (each row corresponds to one joint, columns for x, y, z):")
print(all_19_points_mae_transpose)

# 计算各轴平均 MAE（沿 0 轴取均值），为一个 1x3 的结果
avg_19_points_mae_xyz = np.mean(all_19_points_mae, axis=1).reshape(1, 3)
print("Average MAE for each axis (x, y, z):")
# print(avg_19_points_mae_xyz)

# 分别输出三个轴的总体 MAE（所有关节综合的误差）
mae_x_total = metrics.mean_absolute_error(labels_test[:, 0:19], predictions[:, 0:19])
mae_y_total = metrics.mean_absolute_error(labels_test[:, 19:38], predictions[:, 19:38])
mae_z_total = metrics.mean_absolute_error(labels_test[:, 38:57], predictions[:, 38:57])
print("MAE for x is", mae_x_total)
print("MAE for y is", mae_y_total)
print("MAE for z is", mae_z_total)

# ---------------- 新指标 ----------------

# 1. 计算19个关节每个关节误差（x、y、z误差取平均），得到 19 维向量
avg_joint_error = np.mean(all_19_points_mae_transpose, axis=1)  # shape (19,)
print("Average error for each joint (x, y, z average):")
print(avg_joint_error)

# 2. 计算总体 RMSE（均方根误差）：分别计算 x, y, z 轴所有19个关节的 RMSE
rmse_x = np.sqrt(metrics.mean_squared_error(labels_test[:, 0:19], predictions[:, 0:19]))
rmse_y = np.sqrt(metrics.mean_squared_error(labels_test[:, 19:38], predictions[:, 19:38]))
rmse_z = np.sqrt(metrics.mean_squared_error(labels_test[:, 38:57], predictions[:, 38:57]))
print("Overall RMSE for x is", rmse_x)
print("Overall RMSE for y is", rmse_y)
print("Overall RMSE for z is", rmse_z)

# 3. 计算每个关节每个轴的 RMSE（19x3矩阵）
joint_rmse_x = np.sqrt(np.mean((labels_test[:, 0:19] - predictions[:, 0:19]) ** 2, axis=0))
joint_rmse_y = np.sqrt(np.mean((labels_test[:, 19:38] - predictions[:, 19:38]) ** 2, axis=0))
joint_rmse_z = np.sqrt(np.mean((labels_test[:, 38:57] - predictions[:, 38:57]) ** 2, axis=0))
# 组合为 (19, 3) 形状的矩阵，其中每行依次对应：关节i的RMSE_x, RMSE_y, RMSE_z
joint_rmse_matrix = np.vstack([joint_rmse_x, joint_rmse_y, joint_rmse_z]).T
print("Per-joint RMSE for x, y, z (shape: 19x3):")
print(joint_rmse_matrix)

# ---------------- 保存结果到同一个 Excel 文件（单 Sheet） ----------------

# 构造汇总表（Summary）：使用单个 DataFrame 保存指标，每行一项指标
# 对于向量（如 avg_19_points_mae_xyz 和 avg_joint_error），以逗号分隔的字符串形式保存
summary_data = {
    "Metric": [
        "MAE_x (total)",
        "MAE_y (total)",
        "MAE_z (total)",
        "Average MAE for each axis (x,y,z)",  # 平均 MAE 的 1x3 向量
        "Average error for each joint (x,y,z avg) - 19 values",
        "Overall RMSE_x",
        "Overall RMSE_y",
        "Overall RMSE_z",
        "Average inference time per frame (sec)"
    ],
    "Value": [
        mae_x_total,
        mae_y_total,
        mae_z_total,
        ", ".join(["{:.4f}".format(v) for v in avg_19_points_mae_xyz.flatten()]),
        ", ".join(["{:.4f}".format(v) for v in avg_joint_error]),
        rmse_x,
        rmse_y,
        rmse_z,
        avg_inference_time_per_frame
    ]
}
df_summary = pd.DataFrame(summary_data)

# 构造 19*3 的 Joint_RMSE 表，增加关节编号
df_joint_rmse = pd.DataFrame(joint_rmse_matrix, columns=['RMSE_x', 'RMSE_y', 'RMSE_z'])
df_joint_rmse.insert(0, 'Joint Index', range(1, 20))  # 关节编号从 1 到 19

# 将两个表合并写入单个工作表中：先写入汇总表，再空一行，再写入关节 RMSE 表
excel_output_path = 'output_mmpose-cnn/mmpose-cnn_result.xlsx'
with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
    # 写入汇总表
    df_summary.to_excel(writer, sheet_name='Result', index=False, startrow=0)
    
    # 计算汇总表写入后的行数，预留一行空白
    start_row = df_summary.shape[0] + 2
    # 写入 Joint_RMSE 表
    df_joint_rmse.to_excel(writer, sheet_name='Result', index=False, startrow=start_row)

print("All results have been saved to:", excel_output_path)

Model: "model_16"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_xy (InputLayer)           [(None, 16, 16, 3)]  0                                            
__________________________________________________________________________________________________
input_xz (InputLayer)           [(None, 16, 16, 3)]  0                                            
__________________________________________________________________________________________________
conv2d_108 (Conv2D)             (None, 16, 16, 16)   448         input_xy[0][0]                   
__________________________________________________________________________________________________
conv2d_111 (Conv2D)             (None, 16, 16, 16)   448         input_xz[0][0]                   
___________________________________________________________________________________________

In [ ]:
# import numpy as np 
# # from keras.models import load_model
# from tensorflow.keras.models import load_model
# from keras.optimizers import Adam
# import tensorflow as tf
# from sklearn import metrics

# # 加载模型时传递自定义指标
# model_path = 'output_mmpose-cnn/mmpose-cnn.h5' 
# loaded_model = load_model(
#     model_path,
#     compile = False
# )


# # 验证模型加载成功
# loaded_model.summary()

# # 2. 加载测试数据

# # 预处理所有数据
# feature_test = np.load('reshaped_data_mmPoseCNN/feature_test.npy')
# featuremap_test = preprocess_all_data(feature_test)

# # 分离为两个输入分支
# xy_test = featuremap_test[:, 0]
# xz_test = featuremap_test[:, 1]
# labels_test = np.load('reshaped_data_mmPoseCNN/labels_test.npy')

# # 3. 使用模型进行预测
# predictions = loaded_model.predict([xy_test, xz_test], batch_size=64, verbose=0)


# # 4. 保存预测结果到.npy文件
# filtered_result_path = 'output_mmpose-cnn/mmpose-cnn_result.npy'
# np.save(filtered_result_path, predictions)

# # matrix transformation for the final all 19 points mae
# x_mae = metrics.mean_absolute_error(labels_test[:,0:19], predictions[:,0:19], multioutput = 'raw_values')
# y_mae = metrics.mean_absolute_error(labels_test[:,19:38], predictions[:,19:38], multioutput = 'raw_values')
# z_mae = metrics.mean_absolute_error(labels_test[:,38:57], predictions[:,38:57], multioutput = 'raw_values')

# all_19_points_mae = np.concatenate((x_mae, y_mae, z_mae)).reshape(3,19)
# all_19_points_mae_Transpose = all_19_points_mae.T
# print(all_19_points_mae_Transpose)

# avg_19_points_mae = np.mean(all_19_points_mae, axis = 0)
# avg_19_points_mae_xyz = np.mean(all_19_points_mae, axis = 1).reshape(1,3)
# print(avg_19_points_mae_xyz)

# # 每轴的MAE error for each axis
# print("mae for x is",metrics.mean_absolute_error(labels_test[:,0:19], predictions[:,0:19]))
# print("mae for y is",metrics.mean_absolute_error(labels_test[:,19:38], predictions[:,19:38]))
# print("mae for z is",metrics.mean_absolute_error(labels_test[:,38:57], predictions[:,38:57]))